# 04 Limpieza 01 — Catálogo de afiliaciones UNAM con sedes/campus

Esta libreta crea en memoria un catálogo/diccionario de organismos UNAM usando `UNAM_Completo_Corregido.xlsx` como referencia.

In [1]:
from pathlib import Path
import re
import unicodedata
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, List, Tuple

import pandas as pd

# ---------------------------------------------------------------------
# 1. Configuración de rutas
# ---------------------------------------------------------------------
RUTA_EXCEL_REFERENCIA = Path(
    r"C:\Users\hazar\Desktop\PROYECTO\00_control\UNAM_Completo_Corregido.xlsx"
)

CARPETA_TRABAJO = Path(
    r"C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\01_Normalizar_Afiliaciones"
)


if not RUTA_EXCEL_REFERENCIA.exists():
    alternativa = Path("/mnt/data/UNAM_Completo_Corregido.xlsx")
    if alternativa.exists():
        RUTA_EXCEL_EFECTIVA = alternativa
    else:
        RUTA_EXCEL_EFECTIVA = RUTA_EXCEL_REFERENCIA
else:
    RUTA_EXCEL_EFECTIVA = RUTA_EXCEL_REFERENCIA

CARPETA_TRABAJO.mkdir(parents=True, exist_ok=True)

# Crear el catálogo en memoria, no generar documentos.
EXPORTAR_CATALOGOS = True

print("Excel de referencia:", RUTA_EXCEL_EFECTIVA)
print("Carpeta de trabajo:", CARPETA_TRABAJO)
print("Exportar catálogos:", EXPORTAR_CATALOGOS)

# ---------------------------------------------------------------------
# 2. Funciones básicas de texto
# ---------------------------------------------------------------------
def sin_acentos(texto: Any) -> str:
    texto = "" if texto is None else str(texto)
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


def norm(texto: Any) -> str:
    """Normalización interna para comparar. No se usa para escribir salida final."""
    texto = sin_acentos(texto).lower()
    texto = texto.replace("&", " and ")
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


def limpiar_texto_base(texto: Any) -> str:
    texto = "" if texto is None else str(texto)
    texto = texto.replace("\r", " ").replace("\n", " ")
    texto = re.sub(r"\s+", " ", texto).strip(" ;,|")
    texto = texto.replace("Instituo ", "Instituto ")
    texto = texto.replace("instituo ", "instituto ")
    texto = texto.replace("Facultad deMedicina", "Facultad de Medicina")
    texto = texto.replace("facultad deMedicina", "Facultad de Medicina")
    return texto.strip()


def deduplicar_preservando_orden(valores: Iterable[Any]) -> List[str]:
    salida = []
    vistos = set()
    for valor in valores:
        valor = limpiar_texto_base(valor)
        clave = norm(valor)
        if valor and clave and clave not in vistos:
            salida.append(valor)
            vistos.add(clave)
    return salida

# ---------------------------------------------------------------------
# 3. Sedes/campus/unidades UNAM que NO deben descartarse
# ---------------------------------------------------------------------

SEDES_CANONICAS = {
    "juriquilla": "Juriquilla",
    "cuernavaca": "Cuernavaca",
    "yucatan": "Yucatán",
    "yucatán": "Yucatán",
    "merida": "Mérida",
    "mérida": "Mérida",
    "morelia": "Morelia",
    "leon": "León",
    "león": "León",
    "ensenada": "Ensenada",
    "ciudad universitaria": "Ciudad Universitaria",
}

# Algunos textos usan "Unidad" y otros "Campus". El catálogo conserva la forma
# cuando es informativa.
PATRON_SEDE = re.compile(
    r"\b(?:(Campus|Unidad)\s+)?"
    r"(Juriquilla|Cuernavaca|Yucat[aá]n|M[eé]rida|Morelia|Le[oó]n|Ensenada|Ciudad Universitaria)\b",
    flags=re.I,
)


def normalizar_nombre_sede(sede: Any) -> str:
    return SEDES_CANONICAS.get(norm(sede), limpiar_texto_base(sede))


def detectar_sede_unidad(texto: Any) -> str:
    texto_limpio = limpiar_texto_base(texto)
    if not texto_limpio:
        return ""
    coincidencias = list(PATRON_SEDE.finditer(texto_limpio))
    if not coincidencias:
        return ""
    # Tomar la última mención; suele ser la más específica al final del organismo.
    m = coincidencias[-1]
    prefijo = (m.group(1) or "").strip().lower()
    sede = normalizar_nombre_sede(m.group(2))
    if prefijo == "campus":
        return f"Campus {sede}"
    if prefijo == "unidad":
        return f"Unidad {sede}"
    # Para organismos que oficialmente no usan "Unidad" pero aparecen con ciudad/sede.
    return sede


def base_sin_sede(texto: Any) -> str:
    """Quita sólo el sufijo de sede para obtener una base de comparación, no para salida final."""
    texto = limpiar_texto_base(texto)
    texto = re.sub(
        r"\s*,?\s+(Campus|Unidad)\s+(Juriquilla|Cuernavaca|Yucat[aá]n|M[eé]rida|Morelia|Le[oó]n|Ensenada)$",
        "",
        texto,
        flags=re.I,
    )
    texto = re.sub(
        r"\s*,?\s+(Juriquilla|Cuernavaca|Yucat[aá]n|M[eé]rida|Morelia|Le[oó]n|Ensenada)$",
        "",
        texto,
        flags=re.I,
    )
    return limpiar_texto_base(texto)

Excel de referencia: C:\Users\hazar\Desktop\PROYECTO\00_control\UNAM_Completo_Corregido.xlsx
Carpeta de trabajo: C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\01_Normalizar_Afiliaciones
Exportar catálogos: True


In [2]:
# ---------------------------------------------------------------------
# 4. Catálogos base, genéricos, externos y correcciones canónicas
# ---------------------------------------------------------------------
# Lista inicial basada en la base corregida y en las reglas del proyecto.

ORGANISMOS_AUTORIDAD_INICIAL = [
    "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas",
    "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán",
    "Facultad de Ingeniería",
    "Instituto de Ingeniería",
    "Facultad de Ciencias",
    "Facultad de Ciencias, Unidad Cuernavaca",
    "Facultad de Ciencias, Unidad Yucatán",
    "Instituto de Ciencias Aplicadas y Tecnología",
    "Instituto de Matemáticas",
    "Instituto de Biotecnología",
    "Instituto de Biotecnología, Cuernavaca",
    "Instituto de Ciencias Nucleares",
    "Facultad de Medicina",
    "Instituto de Física",
    "Facultad de Estudios Superiores Cuautitlán",
    "Instituto de Ecología",
    "Centro de Ciencias de la Complejidad",
    "Facultad de Química",
    "Instituto de Geofísica",
    "Instituto de Ciencias de la Atmósfera y Cambio Climático",
    "Centro de Ciencias Genómicas",
    "Centro de Ciencias Genómicas, Cuernavaca",
    "Instituto de Investigaciones Biomédicas",
    "Instituto de Astronomía",
    "Instituto de Geología",
    "Instituto de Energías Renovables",
    "Instituto de Fisiología Celular",
    "Instituto de Biología",
    "Instituto de Investigaciones en Materiales",
    "Escuela Nacional de Estudios Superiores Unidad Morelia",
    "Escuela Nacional de Estudios Superiores Unidad Juriquilla",
    "Escuela Nacional de Estudios Superiores Unidad León",
    "Escuela Nacional de Estudios Superiores Unidad Mérida",
    "Coordinación de Investigación Científica",
    "Instituto de Química",
    "Instituto de Química, Unidad Mérida",
    "Centro de Geociencias",
    "Facultad de Psicología",
    "Instituto de Neurobiología",
    "Instituto de Neurobiología, Unidad Juriquilla",
    "Instituto de Ciencias Físicas",
    "Facultad de Estudios Superiores Iztacala",
    "Facultad de Estudios Superiores Aragón",
    "Centro de Investigaciones en Geografía Ambiental",
    "Instituto de Ciencias del Mar y Limnología",
    "Facultad de Estudios Superiores Zaragoza",
    "Facultad de Medicina Veterinaria y Zootecnia",
    "Instituto de Investigaciones Filosóficas",
    "Centro de Nanociencias y Nanotecnología",
    "Dirección General de Cómputo y de Tecnologías de Información y Comunicación",
    "Instituto de Investigaciones en Ecosistemas y Sustentabilidad",
    "Coordinación de Universidad Abierta, Innovación Educativa y Educación a Distancia",
    "Facultad de Estudios Superiores Acatlán",
    "Centro Regional de Investigaciones Multidisciplinarias",
    "Centro Regional de Investigaciones Multidisciplinarias, Cuernavaca",
    "Centro de Física Aplicada y Tecnología Avanzada",
    "Centro de Física Aplicada y Tecnología Avanzada, Juriquilla",
    "Centro de Ciencias Matemáticas",
    "Centro de Investigaciones Interdisciplinarias en Ciencias y Humanidades",
    "Centro de Investigaciones de Diseño Industrial",
    "Centro de Estudios en Computación Avanzada",
    "Instituto de Radioastronomía y Astrofísica",
    "Facultad de Ciencias Políticas y Sociales",
    "Facultad de Contaduría y Administración",
    "Facultad de Economía",
]

AFILIACIONES_GENERICAS_UNAM = {
    "UNAM",
    "U.N.A.M.",
    "Universidad Nacional Autónoma de México",
    "Universidad Nacional Autonoma de Mexico",
    "Universidad Nacional Autónoma de México (UNAM)",
    "Universidad Nacional Autónoma de México, Campus Juriquilla",
    "Universidad Nacional Autonoma de Mexico Campus Juriquilla",
    "National Autonomous University of Mexico",
    "National Autonomous University Mexico",
}

# No son necesariamente falsas, pero no deben pasar como afiliación final si no aparece una dependencia.
ENTIDADES_NO_FINALES_O_REVISION = {
    "Posgrado en Ciencia e Ingeniería de la Computación",
    "Posgrado en Ingeniería",
    "Posgrado en Ciencias Biologicas",
    "Posgrado en Ciencias Biológicas",
    "Unidad de Posgrado",
    "Licenciatura en Derecho",
    "Licenciatura en Arquitectura",
    "Programa de Posgrado",
    "Programa de Maestría",
    "Programa de Doctorado",
}


CORRECCIONES_CANONICAS = {
    "universidad nacional autonoma de mexico campus juriquilla": "Universidad Nacional Autónoma de México, Campus Juriquilla",
    "universidad nacional autonoma de mexico juriquilla": "Universidad Nacional Autónoma de México, Campus Juriquilla",

    "centro regional de investigaciones multidisciplinarias cuernavaca": "Centro Regional de Investigaciones Multidisciplinarias, Cuernavaca",
    "centro regional de investigaciones multidisciplinarias": "Centro Regional de Investigaciones Multidisciplinarias",

    "centro de estudios de computacion avanzada": "Centro de Estudios en Computación Avanzada",
    "centro de estudios en computacion avanzada": "Centro de Estudios en Computación Avanzada",

    "centro de fisica aplicada y tecnologia avanzada juriquilla": "Centro de Física Aplicada y Tecnología Avanzada, Juriquilla",
    "centro de fisica aplicada y tecnologia avanzada campus juriquilla": "Centro de Física Aplicada y Tecnología Avanzada, Juriquilla",
    "centro de fisica aplicada y tecnologia avanzada": "Centro de Física Aplicada y Tecnología Avanzada",

    "facultad de ciencias politicas y sociales": "Facultad de Ciencias Políticas y Sociales",
    "facultad de contaduria y administracion": "Facultad de Contaduría y Administración",
    "facultad de economia": "Facultad de Economía",
    "facultad demedicina": "Facultad de Medicina",
    "facultad de medicina": "Facultad de Medicina",
    "facultad de ciencias unidad cuernavaca": "Facultad de Ciencias, Unidad Cuernavaca",
    "facultad de ciencias cuernavaca": "Facultad de Ciencias, Unidad Cuernavaca",
    "facultad de ciencias unidad yucatan": "Facultad de Ciencias, Unidad Yucatán",
    "facultad de ciencias yucatan": "Facultad de Ciencias, Unidad Yucatán",
    "facultad de ciencias": "Facultad de Ciencias",

    "instituo de ciencias aplicadas y tecnologia": "Instituto de Ciencias Aplicadas y Tecnología",
    "instituto de ciencias aplicadas y tecnologia": "Instituto de Ciencias Aplicadas y Tecnología",
    "instituo de fisiologia celular": "Instituto de Fisiología Celular",
    "instituto de fisiologia celular": "Instituto de Fisiología Celular",

    "instituto neurobiologia juriquilla": "Instituto de Neurobiología, Unidad Juriquilla",
    "instituto de neurobiologia": "Instituto de Neurobiología",
    "instituto de neurobiologia juriquilla": "Instituto de Neurobiología, Unidad Juriquilla",
    "instituto de neurobiologia unidad juriquilla": "Instituto de Neurobiología, Unidad Juriquilla",

    "instituto de biologia": "Instituto de Biología",
    "instituto de biotecnologia": "Instituto de Biotecnología",
    "instituto de biotecnologia cuernavaca": "Instituto de Biotecnología, Cuernavaca",
    "instituto de biotecnologia juriquilla": "Instituto de Biotecnología, Juriquilla",

    "instituto de ciencias genomicas cuernavaca": "Centro de Ciencias Genómicas, Cuernavaca",
    "centro de ciencias genomicas cuernavaca": "Centro de Ciencias Genómicas, Cuernavaca",
    "centro de ciencias genomicas": "Centro de Ciencias Genómicas",

    "instituto de energias renovables": "Instituto de Energías Renovables",
    "instituto de investigaciones historicas": "Instituto de Investigaciones Históricas",
    "instituto de investigaciones juridicas": "Instituto de Investigaciones Jurídicas",

    "instituto de investigaciones en matematicas aplicadas y en sistemas unidad yucatan": "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán",
    "instituto de investigaciones en matematicas aplicadas y en sistemas yucatan": "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán",
    "unidad academica del iimas en yucatan": "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán",
    "unidad academica iimas yucatan": "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán",
    "iimas unidad yucatan": "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán",
    "iimas yucatan": "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán",
    "instituto de investigaciones en matematicas aplicadas y en sistemas": "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas",

    "instituto de quimica unidad merida": "Instituto de Química, Unidad Mérida",
    "instituto de quimica merida": "Instituto de Química, Unidad Mérida",
    "instituto de quimica": "Instituto de Química",

    "instituto de radioastronomia y astrofisica": "Instituto de Radioastronomía y Astrofísica",
    "direccion general de computo y de tecnologias de informacion y comunicacion": "Dirección General de Cómputo y de Tecnologías de Información y Comunicación",
    "direccion general de computo y tecnologias de informacion y comunicacion": "Dirección General de Cómputo y de Tecnologías de Información y Comunicación",
    "centro de ciencias de la complejidad c3": "Centro de Ciencias de la Complejidad",
    "centro de ciencias de complejidad": "Centro de Ciencias de la Complejidad",
}

# Externos observados o frecuentes.
EXTERNOS_EXACTOS = {
    "tecnologico de monterrey",
    "tecnologico de monterrey campus estado de mexico",
    "el colegio nacional",
    "hospital general de mexico",
    "instituto nacional de cancerologia",
    "instituto nacional de ciencias medicas y nutricion salvador zubiran",
    "instituto nacional de medicina genomica",
    "centro de investigaciones y estudios superiores en antropologia social",
    "centro de investigacion y de estudios avanzados",
    "cinvestav",
    "instituto politecnico nacional",
    "ipn",
    "universidad autonoma metropolitana",
    "uam",
}

EXTERNOS_FRAGMENTOS = [
    "tecnologico de monterrey",
    "monterrey institute of technology",
    "universidad veracruzana",
    "hospital general de mexico",
    "instituto nacional",
    "instituto mexicano del seguro social",
    "imss",
    "el colegio nacional",
    "colegio nacional",
    "university of greenwich",
    "greenwich",
    "asociacion mexicana",
    "ciesas",
    "centro de investigaciones y estudios superiores en antropologia social",
    "instituto de materiales de misiones",
    "universidad autonoma metropolitana",
    "cinvestav",
    "instituto politecnico nacional",
    " ipn ",
]

PATRONES_PLACEHOLDER_AFILIACION = [
    "afiliacion externa",
    "externa no unam",
    "externo no unam",
    "no unam en la fuente consultada",
    "afiliacion no recuperada",
    "no recuperada",
    "no disponible",
    "sin afiliacion",
    "sin informacion de afiliacion",
    "exacta no exportada",
    "no exportada",
    "affiliation not available",
    "affiliation not exported",
    "affiliation not recovered",
    "external affiliation",
]


def es_placeholder_afiliacion(texto: Any) -> bool:
    n = norm(texto)
    if not n:
        return True
    return any(patron in n for patron in PATRONES_PLACEHOLDER_AFILIACION)


def tiene_senal_unam_generica(texto: Any) -> bool:
    n = norm(texto)
    if not n or es_placeholder_afiliacion(texto):
        return False
    patrones = [
        r"\bunam\b",
        r"u n a m",
        r"universidad nacional autonoma de mexico",
        r"national autonomous university of mexico",
        r"national autonomous university mexico",
        r"univ nacl autonoma mex",
        r"univ nacional autonoma de mexico",
        r"ciudad universitaria",
        r"grid 9486 3",
        r"ror org 01tmp8f25",
    ]
    return any(re.search(p, n) for p in patrones)


def es_generico_unam(texto: Any) -> bool:
    n = norm(texto)
    genericos = {norm(v) for v in AFILIACIONES_GENERICAS_UNAM}
    if n in genericos:
        return True
    # UNAM + campus, pero sin dependencia concreta.
    if n.startswith("universidad nacional autonoma de mexico"):
        palabras_dependencia = [
            "instituto", "facultad", "centro", "escuela", "direccion", "coordinacion", "laboratorio"
        ]
        return not any(p in n for p in palabras_dependencia)
    if n.startswith("national autonomous university of mexico"):
        palabras_dependencia = [
            "institute", "faculty", "center", "centre", "school", "department", "laboratory"
        ]
        return not any(p in n for p in palabras_dependencia)
    return False


def es_externo_o_no_objetivo(texto: Any) -> bool:
    n0 = norm(texto)
    n = f" {n0} "
    if not n.strip():
        return True
    if n0 in EXTERNOS_EXACTOS:
        return True
    # Si tiene señal UNAM, no lo marcamos externo aunque contenga un colaborador externo en la misma celda.
    if tiene_senal_unam_generica(texto):
        return False
    return any(fragmento.strip() in n for fragmento in EXTERNOS_FRAGMENTOS)

# ---------------------------------------------------------------------
# 5. Patrones para programas y organismos
# ---------------------------------------------------------------------
PROGRAMA_RE = re.compile(
    r"\b(posgrado|postgrado|licenciatura|maestr[ií]a|doctorado|programa(?:\s+de)?|"
    r"ciencias biom[eé]dicas|unidad de posgrado|graduate program|bachelor|master|phd)\b",
    flags=re.I,
)

EXTRAE_PROGRAMA_RE = re.compile(
    r"(Posgrado en [^,;|]+|"
    r"Postgrado en [^,;|]+|"
    r"Licenciatura en [^,;|]+|"
    r"Maestr[ií]a en [^,;|]+|"
    r"Doctorado en [^,;|]+|"
    r"Programa(?: de)? [^,;|]+|"
    r"Graduate Program(?: in)? [^,;|]+|"
    r"Bachelor(?:'s)?(?: in)? [^,;|]+|"
    r"Master(?:'s)?(?: in)? [^,;|]+|"
    r"Ciencias Biom[eé]dicas|"
    r"Unidad de Posgrado)",
    flags=re.I,
)

ORGANISMO_INICIO_RE = re.compile(
    r"^(universidad nacional|facultad de|instituto de|centro de|centro regional de|"
    r"escuela nacional|direcci[oó]n general|direcci[oó]n de|coordinaci[oó]n de|"
    r"unidad de alta tecnolog[ií]a|laboratorio nacional|laboratorio internacional)",
    flags=re.I,
)

# La parte final permite capturar ", Unidad Yucatán" o ", Campus Juriquilla"
# aunque el nombre institucional base termine antes de la coma.
SUFIJO_SEDE_RE = r"(?:\s*,?\s+(?:Campus|Unidad)\s+(?:Juriquilla|Cuernavaca|Yucat[aá]n|M[eé]rida|Morelia|Le[oó]n|Ensenada)|\s*,?\s+(?:Juriquilla|Cuernavaca|Yucat[aá]n|M[eé]rida|Morelia|Le[oó]n|Ensenada))?"

EXTRAE_ORGANISMO_RE = re.compile(
    r"(Universidad Nacional Aut[oó]noma de M[eé]xico" + SUFIJO_SEDE_RE + r"|"
    r"Facultad de [^,;|]+?" + SUFIJO_SEDE_RE + r"(?=,|;|\||$)|"
    r"Instituto de [^,;|]+?" + SUFIJO_SEDE_RE + r"(?=,|;|\||$)|"
    r"Centro Regional de [^,;|]+?" + SUFIJO_SEDE_RE + r"(?=,|;|\||$)|"
    r"Centro de [^,;|]+?" + SUFIJO_SEDE_RE + r"(?=,|;|\||$)|"
    r"Escuela Nacional de [^,;|]+?" + SUFIJO_SEDE_RE + r"(?=,|;|\||$)|"
    r"Direcci[oó]n General de [^,;|]+?" + SUFIJO_SEDE_RE + r"(?=,|;|\||$)|"
    r"Direcci[oó]n de [^,;|]+?" + SUFIJO_SEDE_RE + r"(?=,|;|\||$)|"
    r"Coordinaci[oó]n de [^,;|]+?" + SUFIJO_SEDE_RE + r"(?=,|;|\||$)|"
    r"Unidad de Alta Tecnolog[ií]a" + SUFIJO_SEDE_RE + r"|"
    r"Laboratorio Nacional de [^,;|]+?" + SUFIJO_SEDE_RE + r"(?=,|;|\||$))",
    flags=re.I,
)


def es_programa(texto: Any) -> bool:
    return bool(PROGRAMA_RE.search(str(texto or "")))


def extraer_programas(texto: Any) -> List[str]:
    texto = limpiar_texto_base(texto)
    if not texto:
        return []
    return deduplicar_preservando_orden(m.group(1) for m in EXTRAE_PROGRAMA_RE.finditer(texto))


def normalizar_organismo_base(texto: Any) -> str:
    """
    Normaliza un organismo sin borrar sedes/campus/unidades.

    Antes se quitaban sufijos como Juriquilla, Cuernavaca o Unidad Yucatán.
    En esta versión se conservan porque pueden ser clave para reconocer unidades UNAM fuera de CU.
    """
    texto = limpiar_texto_base(texto)
    if not texto:
        return ""

    n = norm(texto)
    if n in CORRECCIONES_CANONICAS:
        return CORRECCIONES_CANONICAS[n]

    # Estandarizar una sede final si aparece como sufijo, pero conservarla.
    # Ejemplo: "Instituto de Neurobiología Unidad Juriquilla" ->
    # "Instituto de Neurobiología, Unidad Juriquilla".
    m = re.search(
        r"\s+(Campus|Unidad)\s+(Juriquilla|Cuernavaca|Yucat[aá]n|M[eé]rida|Morelia|Le[oó]n|Ensenada)$",
        texto,
        flags=re.I,
    )
    if m:
        pref = m.group(1).capitalize()
        sede = normalizar_nombre_sede(m.group(2))
        base = limpiar_texto_base(texto[:m.start()])
        texto = f"{base}, {pref} {sede}"
    else:
        m2 = re.search(
            r"\s+(Juriquilla|Cuernavaca|Yucat[aá]n|M[eé]rida|Morelia|Le[oó]n|Ensenada)$",
            texto,
            flags=re.I,
        )
        if m2 and not texto.lower().startswith("universidad nacional"):
            sede = normalizar_nombre_sede(m2.group(1))
            base = limpiar_texto_base(texto[:m2.start()])
            texto = f"{base}, {sede}"

    n2 = norm(texto)
    return CORRECCIONES_CANONICAS.get(n2, texto)


def organismo_sin_sede_para_base(organismo: Any) -> str:
    return normalizar_organismo_base(base_sin_sede(organismo))


def tipo_organismo(organismo: Any) -> str:
    n = norm(base_sin_sede(organismo))
    if n.startswith("instituto"):
        return "instituto"
    if n.startswith("facultad"):
        return "facultad"
    if n.startswith("centro"):
        return "centro"
    if n.startswith("escuela"):
        return "escuela"
    if n.startswith("direccion"):
        return "direccion_general"
    if n.startswith("coordinacion"):
        return "coordinacion"
    if n.startswith("unidad"):
        return "unidad_academica"
    if n.startswith("laboratorio"):
        return "laboratorio"
    if n.startswith("universidad nacional"):
        return "unam_generica"
    return "otro"


def canon_preferido(valores: Iterable[str]) -> str:
    candidatos = [normalizar_organismo_base(v) for v in valores if normalizar_organismo_base(v)]
    if not candidatos:
        return ""
    conteo = Counter(candidatos)
    def score(v: str) -> Tuple[int, int, int, int]:
        s = conteo[v]
        s += 4 * bool(re.search(r"[áéíóúñÁÉÍÓÚÑ]", v))
        s += 3 * bool(detectar_sede_unidad(v))
        s -= 6 * es_generico_unam(v)
        s -= 2 * (len(v) > 120)
        return (s, len(v), v.count(","), -len(base_sin_sede(v)))
    return sorted(candidatos, key=score, reverse=True)[0]


def canon_programa(valores: Iterable[str]) -> str:
    candidatos = [limpiar_texto_base(v) for v in valores if limpiar_texto_base(v)]
    if not candidatos:
        return ""
    conteo = Counter(candidatos)
    def score(v: str) -> Tuple[int, int]:
        s = conteo[v]
        s += 3 * bool(re.search(r"[áéíóúñÁÉÍÓÚÑ]", v))
        return (s, -len(v))
    return sorted(candidatos, key=score, reverse=True)[0]

In [3]:
# ---------------------------------------------------------------------
# 6. Alias manuales, incluyendo campus/unidades fuera de CU
# ---------------------------------------------------------------------
# alias, organismo, fuente_alias, requiere_senal_unam
ALIAS_MANUALES = [
    ("IIMAS", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", "acronimo", False),
    ("Institute for Applied Mathematics and Systems Research", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", "ingles", False),
    ("Instituto de Investigaciones en Matematicas Aplicadas y en Sistemas", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", "sin_acentos", False),
    ("Unidad Académica del IIMAS", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", "variante", False),
    ("Unidad Academica del IIMAS", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas", "variante", False),
    ("Unidad Académica del IIMAS en Yucatán", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán", "variante_sede", False),
    ("Unidad Academica del IIMAS en Yucatan", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán", "variante_sede", False),
    ("IIMAS Unidad Yucatán", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán", "variante_sede", False),
    ("IIMAS Unidad Yucatan", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán", "variante_sede", False),
    ("IIMAS Yucatán", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán", "variante_sede", False),
    ("IIMAS Yucatan", "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Yucatán", "variante_sede", False),

    ("Facultad de Ingeniería", "Facultad de Ingeniería", "canonico", True),
    ("Facultad de Ingenieria", "Facultad de Ingeniería", "sin_acentos", True),
    ("Faculty of Engineering", "Facultad de Ingeniería", "ingles", True),
    ("FI UNAM", "Facultad de Ingeniería", "acronimo", False),

    ("Instituto de Ingeniería", "Instituto de Ingeniería", "canonico", True),
    ("Instituto de Ingenieria", "Instituto de Ingeniería", "sin_acentos", True),
    ("Institute of Engineering", "Instituto de Ingeniería", "ingles", True),
    ("II UNAM", "Instituto de Ingeniería", "acronimo", False),

    ("Facultad de Ciencias", "Facultad de Ciencias", "canonico", True),
    ("Faculty of Sciences", "Facultad de Ciencias", "ingles", True),
    ("Faculty of Science", "Facultad de Ciencias", "ingles", True),
    ("FC UNAM", "Facultad de Ciencias", "acronimo", False),
    ("Facultad de Ciencias Unidad Cuernavaca", "Facultad de Ciencias, Unidad Cuernavaca", "variante_sede", False),
    ("Facultad de Ciencias Cuernavaca", "Facultad de Ciencias, Unidad Cuernavaca", "variante_sede", False),
    ("Facultad de Ciencias Unidad Yucatán", "Facultad de Ciencias, Unidad Yucatán", "variante_sede", False),
    ("Facultad de Ciencias Unidad Yucatan", "Facultad de Ciencias, Unidad Yucatán", "variante_sede", False),

    ("Instituto de Matemáticas", "Instituto de Matemáticas", "canonico", True),
    ("Instituto de Matematicas", "Instituto de Matemáticas", "sin_acentos", True),
    ("Institute of Mathematics", "Instituto de Matemáticas", "ingles", True),
    ("IM UNAM", "Instituto de Matemáticas", "acronimo", False),

    ("Instituto de Ciencias Aplicadas y Tecnología", "Instituto de Ciencias Aplicadas y Tecnología", "canonico", False),
    ("Instituto de Ciencias Aplicadas y Tecnologia", "Instituto de Ciencias Aplicadas y Tecnología", "sin_acentos", False),
    ("Institute of Applied Sciences and Technology", "Instituto de Ciencias Aplicadas y Tecnología", "ingles", False),
    ("ICAT", "Instituto de Ciencias Aplicadas y Tecnología", "acronimo", False),

    ("Dirección General de Cómputo y de Tecnologías de Información y Comunicación", "Dirección General de Cómputo y de Tecnologías de Información y Comunicación", "canonico", False),
    ("Direccion General de Computo y de Tecnologias de Informacion y Comunicacion", "Dirección General de Cómputo y de Tecnologías de Información y Comunicación", "sin_acentos", False),
    ("DGTIC", "Dirección General de Cómputo y de Tecnologías de Información y Comunicación", "acronimo", False),

    ("Centro de Ciencias de la Complejidad", "Centro de Ciencias de la Complejidad", "canonico", False),
    ("Center for Complexity Sciences", "Centro de Ciencias de la Complejidad", "ingles", False),
    ("C3 UNAM", "Centro de Ciencias de la Complejidad", "acronimo", False),
    ("C3", "Centro de Ciencias de la Complejidad", "acronimo", True),

    ("Centro de Ciencias Matemáticas", "Centro de Ciencias Matemáticas", "canonico", False),
    ("Centro de Ciencias Matematicas", "Centro de Ciencias Matemáticas", "sin_acentos", False),
    ("CCM", "Centro de Ciencias Matemáticas", "acronimo", False),

    ("Centro de Ciencias Genómicas", "Centro de Ciencias Genómicas", "canonico", False),
    ("Centro de Ciencias Genomicas", "Centro de Ciencias Genómicas", "sin_acentos", False),
    ("Centro de Ciencias Genómicas Cuernavaca", "Centro de Ciencias Genómicas, Cuernavaca", "variante_sede", False),
    ("Centro de Ciencias Genomicas Cuernavaca", "Centro de Ciencias Genómicas, Cuernavaca", "variante_sede", False),
    ("CCG", "Centro de Ciencias Genómicas", "acronimo", False),

    ("Centro Regional de Investigaciones Multidisciplinarias", "Centro Regional de Investigaciones Multidisciplinarias", "canonico", False),
    ("Centro Regional de Investigaciones Multidisciplinarias Cuernavaca", "Centro Regional de Investigaciones Multidisciplinarias, Cuernavaca", "variante_sede", False),
    ("CRIM", "Centro Regional de Investigaciones Multidisciplinarias", "acronimo", False),
    ("CRIM Cuernavaca", "Centro Regional de Investigaciones Multidisciplinarias, Cuernavaca", "acronimo_sede", False),

    ("Centro de Física Aplicada y Tecnología Avanzada", "Centro de Física Aplicada y Tecnología Avanzada", "canonico", False),
    ("Centro de Fisica Aplicada y Tecnologia Avanzada", "Centro de Física Aplicada y Tecnología Avanzada", "sin_acentos", False),
    ("Centro de Física Aplicada y Tecnología Avanzada Juriquilla", "Centro de Física Aplicada y Tecnología Avanzada, Juriquilla", "variante_sede", False),
    ("Centro de Fisica Aplicada y Tecnologia Avanzada Juriquilla", "Centro de Física Aplicada y Tecnología Avanzada, Juriquilla", "variante_sede", False),
    ("CFATA", "Centro de Física Aplicada y Tecnología Avanzada", "acronimo", False),
    ("CFATA Juriquilla", "Centro de Física Aplicada y Tecnología Avanzada, Juriquilla", "acronimo_sede", False),

    ("Instituto de Neurobiología", "Instituto de Neurobiología", "canonico", False),
    ("Instituto de Neurobiologia", "Instituto de Neurobiología", "sin_acentos", False),
    ("Instituto de Neurobiología Unidad Juriquilla", "Instituto de Neurobiología, Unidad Juriquilla", "variante_sede", False),
    ("Instituto de Neurobiologia Unidad Juriquilla", "Instituto de Neurobiología, Unidad Juriquilla", "variante_sede", False),
    ("Instituto de Neurobiología Juriquilla", "Instituto de Neurobiología, Unidad Juriquilla", "variante_sede", False),
    ("Instituto de Neurobiologia Juriquilla", "Instituto de Neurobiología, Unidad Juriquilla", "variante_sede", False),
    ("INB Juriquilla", "Instituto de Neurobiología, Unidad Juriquilla", "acronimo_sede", False),

    ("Instituto de Biotecnología", "Instituto de Biotecnología", "canonico", False),
    ("Instituto de Biotecnologia", "Instituto de Biotecnología", "sin_acentos", False),
    ("Instituto de Biotecnología Cuernavaca", "Instituto de Biotecnología, Cuernavaca", "variante_sede", False),
    ("Instituto de Biotecnologia Cuernavaca", "Instituto de Biotecnología, Cuernavaca", "variante_sede", False),
    ("IBt Cuernavaca", "Instituto de Biotecnología, Cuernavaca", "acronimo_sede", False),

    ("Instituto de Química Unidad Mérida", "Instituto de Química, Unidad Mérida", "variante_sede", False),
    ("Instituto de Quimica Unidad Merida", "Instituto de Química, Unidad Mérida", "variante_sede", False),

    ("Centro de Estudios en Computación Avanzada", "Centro de Estudios en Computación Avanzada", "canonico", False),
    ("Centro de Estudios de Computación Avanzada", "Centro de Estudios en Computación Avanzada", "variante", False),
    ("Centro de Estudios en Computacion Avanzada", "Centro de Estudios en Computación Avanzada", "sin_acentos", False),

    ("FES Acatlán", "Facultad de Estudios Superiores Acatlán", "acronimo", False),
    ("FES Acatlan", "Facultad de Estudios Superiores Acatlán", "acronimo_sin_acentos", False),
    ("FES Aragón", "Facultad de Estudios Superiores Aragón", "acronimo", False),
    ("FES Aragon", "Facultad de Estudios Superiores Aragón", "acronimo_sin_acentos", False),
    ("FES Cuautitlán", "Facultad de Estudios Superiores Cuautitlán", "acronimo", False),
    ("FES Cuautitlan", "Facultad de Estudios Superiores Cuautitlán", "acronimo_sin_acentos", False),
    ("FES Iztacala", "Facultad de Estudios Superiores Iztacala", "acronimo", False),
    ("FES Zaragoza", "Facultad de Estudios Superiores Zaragoza", "acronimo", False),

    ("ENES Juriquilla", "Escuela Nacional de Estudios Superiores Unidad Juriquilla", "acronimo", False),
    ("ENES León", "Escuela Nacional de Estudios Superiores Unidad León", "acronimo", False),
    ("ENES Leon", "Escuela Nacional de Estudios Superiores Unidad León", "acronimo_sin_acentos", False),
    ("ENES Morelia", "Escuela Nacional de Estudios Superiores Unidad Morelia", "acronimo", False),
    ("ENES Mérida", "Escuela Nacional de Estudios Superiores Unidad Mérida", "acronimo", False),
    ("ENES Merida", "Escuela Nacional de Estudios Superiores Unidad Mérida", "acronimo_sin_acentos", False),
]

In [4]:
# ---------------------------------------------------------------------
# 7. Lectura de la base de referencia
# ---------------------------------------------------------------------
COLUMNAS_CANONICAS = [
    "indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
    "ISBN", "ISSN", "Doi", "URL", "Area", "Subarea", "Keywords", "Abstract"
]

df_referencia = pd.read_excel(RUTA_EXCEL_EFECTIVA, dtype=str, keep_default_na=False)

faltantes = [c for c in ["Afiliacion1", "Afiliacion2"] if c not in df_referencia.columns]
if faltantes:
    raise ValueError(f"No encontré columnas de afiliación en el Excel: {faltantes}")

print("Filas en base de referencia:", len(df_referencia))
print("Columnas encontradas:", list(df_referencia.columns))

valores_afiliacion = []
for col in ["Afiliacion1", "Afiliacion2"]:
    if col in df_referencia.columns:
        for valor in df_referencia[col].tolist():
            valor = limpiar_texto_base(valor)
            if valor and not es_placeholder_afiliacion(valor):
                valores_afiliacion.append({"columna": col, "valor": valor})

valores_afiliacion_df = pd.DataFrame(valores_afiliacion)
print("Valores de afiliación no vacíos:", len(valores_afiliacion_df))
print("Afiliaciones únicas observadas:", valores_afiliacion_df["valor"].nunique() if not valores_afiliacion_df.empty else 0)

# ---------------------------------------------------------------------
# 8. Extracción de candidatos: organismos, programas, genéricos, externos
# ---------------------------------------------------------------------
organismo_candidatos = []
programa_candidatos = []
genericos_candidatos = []
externos_candidatos = []
revision_candidatos = []

# Organismos de autoridad inicial.
for org in ORGANISMOS_AUTORIDAD_INICIAL:
    organismo_candidatos.append({
        "organismo_original": org,
        "organismo_normalizado": normalizar_organismo_base(org),
        "fuente": "autoridad_inicial",
        "evidencia": "lista_base_proyecto",
    })


def detectar_organismos_en_texto(texto: Any) -> List[Tuple[str, str]]:
    """Devuelve pares (organismo, evidencia) detectados dentro de una afiliación cruda."""
    texto_limpio = limpiar_texto_base(texto)
    if not texto_limpio:
        return []

    encontrados = []

    # 1) Si el texto completo ya parece organismo.
    if ORGANISMO_INICIO_RE.search(texto_limpio):
        org = normalizar_organismo_base(texto_limpio)
        if org:
            encontrados.append((org, "texto_completo"))

    # 2) Extraer patrones institucionales en celdas largas, preservando sedes/unidades.
    for m in EXTRAE_ORGANISMO_RE.finditer(texto_limpio):
        org = normalizar_organismo_base(m.group(1))
        if org:
            encontrados.append((org, "regex_organismo"))

    # 3) Detectar alias manuales dentro del texto.
    n_texto = norm(texto_limpio)
    tiene_unam = tiene_senal_unam_generica(texto_limpio)
    for alias, organismo, fuente_alias, requiere_senal in ALIAS_MANUALES:
        n_alias = norm(alias)
        if not n_alias:
            continue
        if re.search(rf"\b{re.escape(n_alias)}\b", n_texto):
            if requiere_senal and not tiene_unam:
                revision_candidatos.append({
                    "valor": texto_limpio,
                    "motivo": "alias_ambiguo_sin_senal_unam",
                    "alias_detectado": alias,
                    "organismo_sugerido": organismo,
                })
            else:
                encontrados.append((normalizar_organismo_base(organismo), f"alias_manual:{alias}"))

    # 4) Deduplicar.
    salida = []
    vistos = set()
    for org, evidencia in encontrados:
        org = normalizar_organismo_base(org)
        clave = norm(org)
        if clave and clave not in vistos:
            salida.append((org, evidencia))
            vistos.add(clave)
    return salida


for _, row in valores_afiliacion_df.iterrows():
    valor = row["valor"]
    columna = row["columna"]

    if es_placeholder_afiliacion(valor):
        revision_candidatos.append({"valor": valor, "motivo": "placeholder", "alias_detectado": "", "organismo_sugerido": ""})
        continue

    if es_generico_unam(valor):
        genericos_candidatos.append({
            "valor": valor,
            "columna": columna,
            "motivo": "unam_generica" if not detectar_sede_unidad(valor) else "unam_generica_con_sede",
            "sede_unidad": detectar_sede_unidad(valor),
        })
        continue

    if es_externo_o_no_objetivo(valor):
        externos_candidatos.append({"valor": valor, "columna": columna, "motivo": "externa_o_no_objetivo"})
        continue

    programas = extraer_programas(valor)
    organismos = detectar_organismos_en_texto(valor)

    for programa in programas:
        organismo_sugerido = organismos[0][0] if organismos else ""
        programa_candidatos.append({
            "programa_original": programa,
            "programa_normalizado": canon_programa([programa]),
            "organismo_unam_sugerido": organismo_sugerido,
            "evidencia": valor,
            "columna": columna,
        })

    if programas and not organismos:
        revision_candidatos.append({
            "valor": valor,
            "motivo": "programa_sin_organismo_detectado",
            "alias_detectado": "",
            "organismo_sugerido": "",
        })

    for org, evidencia in organismos:
        org_norm = normalizar_organismo_base(org)
        if es_generico_unam(org_norm):
            genericos_candidatos.append({
                "valor": org_norm,
                "columna": columna,
                "motivo": "unam_generica_extraida" if not detectar_sede_unidad(org_norm) else "unam_generica_con_sede_extraida",
                "sede_unidad": detectar_sede_unidad(org_norm),
            })
        elif norm(org_norm) in {norm(x) for x in ENTIDADES_NO_FINALES_O_REVISION}:
            revision_candidatos.append({
                "valor": valor,
                "motivo": "entidad_no_final_o_revision",
                "alias_detectado": evidencia,
                "organismo_sugerido": org_norm,
            })
        elif not es_programa(org_norm) and not es_externo_o_no_objetivo(org_norm):
            organismo_candidatos.append({
                "organismo_original": org,
                "organismo_normalizado": org_norm,
                "fuente": "excel_referencia",
                "evidencia": valor,
            })

Filas en base de referencia: 5255
Columnas encontradas: ['indice', 'Titulo', 'Año', 'Autor_norm', 'Afiliacion1', 'Afiliacion2', 'ISBN', 'ISSN', 'Doi', 'URL', 'Area', 'Subarea', 'Keywords', 'Abstract']
Valores de afiliación no vacíos: 5734
Afiliaciones únicas observadas: 127


In [5]:
# ---------------------------------------------------------------------
# 9. Construcción del catálogo de organismos UNAM válidos
# ---------------------------------------------------------------------
organismos_por_clave = defaultdict(list)
evidencias_por_clave = defaultdict(list)
fuentes_por_clave = defaultdict(set)

for item in organismo_candidatos:
    org = normalizar_organismo_base(item["organismo_normalizado"])
    clave = norm(org)
    if not clave:
        continue
    if es_generico_unam(org):
        genericos_candidatos.append({
            "valor": org,
            "columna": "",
            "motivo": "unam_generica_en_candidatos" if not detectar_sede_unidad(org) else "unam_generica_con_sede_en_candidatos",
            "sede_unidad": detectar_sede_unidad(org),
        })
        continue
    if es_externo_o_no_objetivo(org):
        externos_candidatos.append({"valor": org, "columna": "", "motivo": "externa_en_candidatos"})
        continue
    organismos_por_clave[clave].append(org)
    evidencias_por_clave[clave].append(item.get("evidencia", ""))
    fuentes_por_clave[clave].add(item.get("fuente", ""))

catalogo_rows = []
for clave, valores in organismos_por_clave.items():
    canonico = canon_preferido(valores)
    sede = detectar_sede_unidad(canonico)
    base = organismo_sin_sede_para_base(canonico)
    es_revision = norm(canonico) in {norm(x) for x in ENTIDADES_NO_FINALES_O_REVISION}
    catalogo_rows.append({
        "organismo_unam": canonico,
        "organismo_norm": norm(canonico),
        "organismo_base": base,
        "sede_unidad": sede,
        "tipo_organismo": tipo_organismo(canonico),
        "fuente_catalogo": "; ".join(sorted(f for f in fuentes_por_clave[clave] if f)),
        "conteo_evidencias": len(evidencias_por_clave[clave]),
        "ejemplo_evidencia": next((e for e in evidencias_por_clave[clave] if e), ""),
        "es_valido_final": not es_revision and not es_generico_unam(canonico) and tipo_organismo(canonico) != "unam_generica",
        "requiere_revision": es_revision,
        "motivo_revision": "entidad_no_final_o_demasiado_general" if es_revision else "",
    })

catalogo_organismos_unam_df = pd.DataFrame(catalogo_rows)
if not catalogo_organismos_unam_df.empty:
    catalogo_organismos_unam_df = (
        catalogo_organismos_unam_df
        .drop_duplicates(subset=["organismo_norm"])
        .sort_values(["tipo_organismo", "organismo_base", "sede_unidad", "organismo_unam"])
        .reset_index(drop=True)
    )
else:
    catalogo_organismos_unam_df = pd.DataFrame(columns=[
        "organismo_unam", "organismo_norm", "organismo_base", "sede_unidad", "tipo_organismo",
        "fuente_catalogo", "conteo_evidencias", "ejemplo_evidencia", "es_valido_final",
        "requiere_revision", "motivo_revision"
    ])

ORGANISMOS_VALIDOS = set(
    catalogo_organismos_unam_df.loc[
        catalogo_organismos_unam_df["es_valido_final"].eq(True),
        "organismo_unam"
    ]
)

# ---------------------------------------------------------------------
# 10. Catálogo de programas por asociar
# ---------------------------------------------------------------------
programas_por_clave = defaultdict(list)
programas_organismo_sugerido = defaultdict(Counter)
programas_evidencia = defaultdict(list)

for item in programa_candidatos:
    programa = canon_programa([item["programa_normalizado"]])
    clave = norm(programa)
    if not clave:
        continue
    programas_por_clave[clave].append(programa)
    if item.get("organismo_unam_sugerido"):
        programas_organismo_sugerido[clave][item["organismo_unam_sugerido"]] += 1
    programas_evidencia[clave].append(item.get("evidencia", ""))

programa_rows = []
for clave, valores in programas_por_clave.items():
    programa = canon_programa(valores)
    sugerido = ""
    if programas_organismo_sugerido[clave]:
        sugerido = programas_organismo_sugerido[clave].most_common(1)[0][0]
    programa_rows.append({
        "programa_unam": programa,
        "programa_norm": norm(programa),
        "organismo_unam_sugerido": sugerido,
        "organismo_unam": "",  # se llenará manualmente o en una fase posterior
        "ejemplo_evidencia": next((e for e in programas_evidencia[clave] if e), ""),
    })

catalogo_programas_unam_por_asociar_df = pd.DataFrame(programa_rows)
if not catalogo_programas_unam_por_asociar_df.empty:
    catalogo_programas_unam_por_asociar_df = (
        catalogo_programas_unam_por_asociar_df
        .drop_duplicates(subset=["programa_norm"])
        .sort_values("programa_unam")
        .reset_index(drop=True)
    )

PROGRAMAS_POR_ASOCIAR = set(catalogo_programas_unam_por_asociar_df["programa_unam"]) if not catalogo_programas_unam_por_asociar_df.empty else set()

# ---------------------------------------------------------------------
# 11. Genéricos, externos y revisión
# ---------------------------------------------------------------------
afiliaciones_genericas_unam_df = pd.DataFrame(genericos_candidatos).drop_duplicates() if genericos_candidatos else pd.DataFrame(columns=["valor", "columna", "motivo", "sede_unidad"])
afiliaciones_externas_observadas_df = pd.DataFrame(externos_candidatos).drop_duplicates() if externos_candidatos else pd.DataFrame(columns=["valor", "columna", "motivo"])
afiliaciones_requieren_revision_df = pd.DataFrame(revision_candidatos).drop_duplicates() if revision_candidatos else pd.DataFrame(columns=["valor", "motivo", "alias_detectado", "organismo_sugerido"])

In [6]:
# ---------------------------------------------------------------------
# 12. Diccionario alias -> organismo_unam
# ---------------------------------------------------------------------
alias_rows = []

def add_alias(alias: Any, organismo: Any, fuente_alias: str, requiere_senal_unam: bool = False):
    alias = limpiar_texto_base(alias)
    organismo = normalizar_organismo_base(organismo)
    if not alias or not organismo:
        return
    alias_rows.append({
        "alias": alias,
        "alias_norm": norm(alias),
        "organismo_unam": organismo,
        "organismo_norm": norm(organismo),
        "organismo_base": organismo_sin_sede_para_base(organismo),
        "sede_unidad": detectar_sede_unidad(organismo),
        "fuente_alias": fuente_alias,
        "requiere_senal_unam": bool(requiere_senal_unam),
        "es_alias_sede": bool(detectar_sede_unidad(alias) or detectar_sede_unidad(organismo)),
    })

# Alias derivados del catálogo.
for _, row in catalogo_organismos_unam_df.iterrows():
    organismo = row["organismo_unam"]
    base = row["organismo_base"]
    sede = row["sede_unidad"]

    add_alias(organismo, organismo, "catalogo_canonico", False)
    add_alias(sin_acentos(organismo), organismo, "catalogo_sin_acentos", False)

    # Alias base sólo si no fuerza a confundir sede.

    if sede and base and base != organismo:
        if base in ORGANISMOS_VALIDOS:
            add_alias(base, base, "base_sin_sede", False)
        add_alias(f"{base} {sede}", organismo, "base_mas_sede", False)
        add_alias(f"{base}, {sede}", organismo, "base_mas_sede", False)
    elif base:
        add_alias(base, organismo, "base", False)

# Alias manuales explícitos.
for alias, organismo, fuente_alias, requiere_senal in ALIAS_MANUALES:
    add_alias(alias, organismo, fuente_alias, requiere_senal)

# Resolver conflictos de alias exactos: preferir alias de sede si el alias contiene sede
def score_alias(row: pd.Series) -> Tuple[int, int, int]:
    score = 0
    score += 10 * bool(row.get("es_alias_sede"))
    score += 3 * bool(row.get("fuente_alias", "").startswith("variante_sede"))
    score += 2 * bool(row.get("fuente_alias", "").startswith("acronimo_sede"))
    score += 1 * bool(re.search(r"[áéíóúñÁÉÍÓÚÑ]", str(row.get("organismo_unam", ""))))
    return (score, len(str(row.get("alias", ""))), len(str(row.get("organismo_unam", ""))))

if alias_rows:
    tmp_alias_df = pd.DataFrame(alias_rows)
    # Agrupar por alias_norm y conservar el mejor candidato por score.
    seleccionados = []
    for alias_norm, grupo in tmp_alias_df.groupby("alias_norm", sort=False):
        grupo = grupo.copy()
        grupo["_score"] = grupo.apply(score_alias, axis=1)
        grupo = grupo.sort_values("_score", ascending=False)
        seleccionados.append(grupo.iloc[0].drop(labels=["_score"]).to_dict())
    diccionario_alias_organismos_unam_df = pd.DataFrame(seleccionados)
    diccionario_alias_organismos_unam_df = (
        diccionario_alias_organismos_unam_df
        .sort_values(["es_alias_sede", "organismo_unam", "alias"], ascending=[False, True, True])
        .reset_index(drop=True)
    )
else:
    diccionario_alias_organismos_unam_df = pd.DataFrame(columns=[
        "alias", "alias_norm", "organismo_unam", "organismo_norm", "organismo_base",
        "sede_unidad", "fuente_alias", "requiere_senal_unam", "es_alias_sede"
    ])

ALIAS_A_ORGANISMO = {
    row["alias_norm"]: row["organismo_unam"]
    for _, row in diccionario_alias_organismos_unam_df.iterrows()
    if row["alias_norm"] and row["organismo_unam"]
}

# Alias ordenados por longitud descendente para que "IIMAS Unidad Yucatán" gane sobre "IIMAS".
ALIAS_ORDENADOS = sorted(
    diccionario_alias_organismos_unam_df.to_dict("records"),
    key=lambda r: len(r.get("alias_norm", "")),
    reverse=True,
)

In [7]:
# ---------------------------------------------------------------------
# 13. Funciones de búsqueda para la siguiente fase
# ---------------------------------------------------------------------
def buscar_organismos_unam_en_texto(texto: Any) -> List[Dict[str, Any]]:
    """
    Busca organismos UNAM en una afiliación cruda.
    Devuelve una lista de candidatos ordenados por especificidad.
    No modifica la base.
    """
    texto_limpio = limpiar_texto_base(texto)
    n_texto = norm(texto_limpio)

    if not texto_limpio or es_placeholder_afiliacion(texto_limpio):
        return [{
            "organismo_unam": "",
            "estado": "placeholder_o_vacio",
            "alias_detectado": "",
            "sede_unidad": "",
            "es_valido_final": False,
        }]

    candidatos = []
    tiene_unam = tiene_senal_unam_generica(texto_limpio)

    for item in ALIAS_ORDENADOS:
        alias_norm = item["alias_norm"]
        if not alias_norm:
            continue
        if re.search(rf"\b{re.escape(alias_norm)}\b", n_texto):
            if item.get("requiere_senal_unam") and not tiene_unam:
                continue
            organismo = item["organismo_unam"]
            candidatos.append({
                "organismo_unam": organismo,
                "estado": "organismo_unam_detectado",
                "alias_detectado": item["alias"],
                "sede_unidad": item.get("sede_unidad", ""),
                "es_valido_final": organismo in ORGANISMOS_VALIDOS,
            })

    # Deduplicar por organismo, preservando orden de especificidad.
    salida = []
    vistos = set()
    for c in candidatos:
        clave = norm(c["organismo_unam"])
        if clave and clave not in vistos:
            salida.append(c)
            vistos.add(clave)

    if salida:
        return salida

    if es_programa(texto_limpio) and tiene_unam:
        return [{
            "organismo_unam": "",
            "estado": "programa_unam_sin_organismo_resuelto",
            "alias_detectado": "",
            "sede_unidad": detectar_sede_unidad(texto_limpio),
            "es_valido_final": False,
        }]

    if tiene_unam:
        return [{
            "organismo_unam": "",
            "estado": "unam_generica_sin_organismo",
            "alias_detectado": "",
            "sede_unidad": detectar_sede_unidad(texto_limpio),
            "es_valido_final": False,
        }]

    return [{
        "organismo_unam": "",
        "estado": "externa_o_no_unam",
        "alias_detectado": "",
        "sede_unidad": "",
        "es_valido_final": False,
    }]


def buscar_organismo_unam_en_texto(texto: Any) -> Tuple[str, str]:
    """
    Versión simple para usar en la siguiente libreta.
    Regresa (organismo_unam, estado).
    """
    candidatos = buscar_organismos_unam_en_texto(texto)
    if not candidatos:
        return "", "sin_resultado"
    mejor = candidatos[0]
    return mejor["organismo_unam"], mejor["estado"]

In [8]:
# ---------------------------------------------------------------------
# 14. Resumen y pruebas rápidas
# ---------------------------------------------------------------------
print("\nResumen del catálogo creado en memoria")
print("Organismos catalogados:", len(catalogo_organismos_unam_df))
print("Organismos válidos finales:", len(ORGANISMOS_VALIDOS))
print("Organismos con sede/unidad:", int(catalogo_organismos_unam_df["sede_unidad"].astype(str).str.strip().ne("").sum()))
print("Alias catalogados:", len(diccionario_alias_organismos_unam_df))
print("Alias con sede/unidad:", int(diccionario_alias_organismos_unam_df["es_alias_sede"].sum()) if not diccionario_alias_organismos_unam_df.empty else 0)
print("Programas detectados:", len(catalogo_programas_unam_por_asociar_df))
print("Genéricos UNAM detectados:", len(afiliaciones_genericas_unam_df))
print("Externos observados:", len(afiliaciones_externas_observadas_df))
print("Requieren revisión:", len(afiliaciones_requieren_revision_df))

print("\nOrganismos con sede/unidad detectados en el catálogo:")
cols_sede = ["organismo_unam", "organismo_base", "sede_unidad", "tipo_organismo", "es_valido_final"]
print(catalogo_organismos_unam_df.loc[
    catalogo_organismos_unam_df["sede_unidad"].astype(str).str.strip().ne(""),
    cols_sede
].to_string(index=False))

EJEMPLOS_PRUEBA = [
    "Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas Unidad Yucatán, UNAM",
    "Unidad Académica del IIMAS en Yucatán, Universidad Nacional Autónoma de México",
    "IIMAS Unidad Yucatan",
    "Instituto de Neurobiología Unidad Juriquilla, Universidad Nacional Autónoma de México",
    "Centro de Física Aplicada y Tecnología Avanzada Juriquilla, UNAM",
    "Facultad de Ciencias Unidad Yucatán, UNAM",
    "Centro Regional de Investigaciones Multidisciplinarias Cuernavaca, UNAM",
    "Centro de Ciencias Genómicas Cuernavaca, Universidad Nacional Autónoma de México",
    "Universidad Nacional Autónoma de México Campus Juriquilla",
    "Posgrado en Ciencia e Ingeniería de la Computación, Universidad Nacional Autónoma de México",
    "Tecnológico de Monterrey, Mexico",
]

print("\nPruebas rápidas de búsqueda:")
for ejemplo in EJEMPLOS_PRUEBA:
    organismo, estado = buscar_organismo_unam_en_texto(ejemplo)
    print("-", ejemplo)
    print("  ->", organismo if organismo else "[sin organismo final]", "|", estado)

# ---------------------------------------------------------------------
# 15. Exportación opcional
# ---------------------------------------------------------------------
if EXPORTAR_CATALOGOS:
    ruta_catalogo = CARPETA_TRABAJO / "catalogo_organismos_unam.csv"
    ruta_alias = CARPETA_TRABAJO / "diccionario_alias_organismos_unam.csv"
    ruta_programas = CARPETA_TRABAJO / "catalogo_programas_unam_por_asociar.csv"
    ruta_genericos = CARPETA_TRABAJO / "afiliaciones_genericas_unam.csv"
    ruta_revision = CARPETA_TRABAJO / "afiliaciones_requieren_revision.csv"

    catalogo_organismos_unam_df.to_csv(ruta_catalogo, index=False, encoding="utf-8-sig")
    diccionario_alias_organismos_unam_df.to_csv(ruta_alias, index=False, encoding="utf-8-sig")
    catalogo_programas_unam_por_asociar_df.to_csv(ruta_programas, index=False, encoding="utf-8-sig")
    afiliaciones_genericas_unam_df.to_csv(ruta_genericos, index=False, encoding="utf-8-sig")
    afiliaciones_requieren_revision_df.to_csv(ruta_revision, index=False, encoding="utf-8-sig")

    print("\nCatálogos exportados:")
    print(ruta_catalogo)
    print(ruta_alias)
    print(ruta_programas)
    print(ruta_genericos)
    print(ruta_revision)
else:
    print("\nEXPORTAR_CATALOGOS = False. No se generaron archivos; el catálogo queda en memoria.")


Resumen del catálogo creado en memoria
Organismos catalogados: 88
Organismos válidos finales: 88
Organismos con sede/unidad: 15
Alias catalogados: 129
Alias con sede/unidad: 28
Programas detectados: 8
Genéricos UNAM detectados: 3
Externos observados: 11
Requieren revisión: 18

Organismos con sede/unidad detectados en el catálogo:
                                                                     organismo_unam                                                      organismo_base       sede_unidad tipo_organismo  es_valido_final
                 Centro Regional de Investigaciones Multidisciplinarias, Cuernavaca              Centro Regional de Investigaciones Multidisciplinarias        Cuernavaca         centro             True
                                           Centro de Ciencias Genómicas, Cuernavaca                                        Centro de Ciencias Genómicas        Cuernavaca         centro             True
                        Centro de Física Aplicada y Tecnologí